In [1]:
import pandas as pd

# Load
covariates = pd.read_csv("Data/covariates_deployment_dataset_2026-03-17.csv")
correction = pd.read_csv("Input_phase_2/Correction_crm_users.csv")
upload = pd.read_csv("Input_phase_2/treatment_selected_binary_2.csv")

# Merge upload with covariates
df = upload.merge(covariates, on="customer_nk", how="inner")

# Flag customers present in correction file
corrected_ids = correction["user_id"].drop_duplicates()
df["in_correction"] = df["customer_nk"].isin(corrected_ids).astype(int)

C:\Users\tsterk\AppData\Local\Temp\ipykernel_24196\1264646847.py:4: DtypeWarning: Columns (0: total_volume, 1: food_total, 2: sports_total, 3: monetary_value_52wk, 4: online_sales_52w, 5: retail_sales_52w, 6: monetary_value_53w_104w, 7: online_sales_53w_104w, 8: retail_sales_53w_104w) have mixed types. Specify dtype option on import or set low_memory=False.
  covariates = pd.read_csv("Data/covariates_deployment_dataset_2026-03-17.csv")


In [2]:
df.head()

,customer_nk,original_incentive_name,has_rfl,gender,country_sk,recency,frequency,monetary_value,total_volume,length_of_relationship,...,monetary_value_52wk,volume_52wk,online_sales_52w,retail_sales_52w,frequency_53w_104w,monetary_value_53w_104w,volume_53w_104w,online_sales_53w_104w,retail_sales_53w_104w,in_correction
0,2ae5b545-1b2e-4884-94b8-3cc0ddba570f,BNLX_ChurnP_10_test_export.csv,1,M,hbi|eu|nl,434,45,632.11,175,"1,855",...,132.43,32,0,132.43,10,167.5,39,0,167.5,1
1,d4af981c-08b0-4571-9437-c542dd72314b,BNLX_ChurnP_10_test_export.csv,1,M,hbi|eu|be,410,56,768.7,200,"1,865",...,94.38,19,0,94.38,13,107.91,31,0,107.91,0
2,147e0157-62bb-4c86-8b58-e4104273ba90,BNLX_ChurnP_10_test_export.csv,1,F,hbi|eu|nl,559,23,521.57,165,"1,825",...,69.29,23,0,69.29,8,102.84,48,0,102.84,1
3,c5ce6e19-f417-40b6-a391-348a06b35e94,BNLX_ChurnP_10_test_export.csv,1,F,hbi|eu|nl,505,21,"1,107.85",132,"1,806",...,207.38,14,149.82,57.56,4,166.24,19,106.79,59.45,1
4,125d7fa6-dea7-455f-87ca-cd88ffdff391,BNLX_ChurnP_10_test_export.csv,1,F,hbi|eu|be,389,29,924.23,246,"1,848",...,160.58,39,36.98,123.6,9,348.1,80,0,348.1,1


In [3]:
def incentive_distribution_comparison(df: pd.DataFrame) -> pd.DataFrame:
    before = (
        df["original_incentive_name"]
        .value_counts(normalize=False)
        .rename("before")
    )
    
    after = (
        df.loc[df["in_correction"] == 1, "original_incentive_name"]
        .value_counts(normalize=False)
        .rename("after")
    )
    
    return (
        pd.concat([before, after], axis=1)
        .fillna(0)
        .reset_index()
        .rename(columns={"index": "original_incentive_name"})
        .sort_values("before", ascending=False)
    )


df_incentive_dist = incentive_distribution_comparison(df)

df_incentive_dist

,original_incentive_name,before,after
0,BNLX_ChurnP_10_test_export.csv,2970,2349
1,BNLX_ChurnP_10eu_test_export.csv,2970,2379
2,BNLX_ChurnP_250_test_export.csv,2970,2401
3,BNLX_ChurnP_25_test_export.csv,2970,2406
4,BNLX_ChurnP_500_test_export.csv,2970,2445
5,BNLX_ChurnP_5eu_test_export.csv,2970,2496
6,BNLX_ChurnP_SKUe_test_export.csv,2970,2372


In [4]:
def coerce_metrics_to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()

    df[cols] = (
        df[cols]
        .replace({",": ""}, regex=True)
        .apply(pd.to_numeric, errors="coerce")
    )
    return df

In [5]:
def compute_means_comparison(df: pd.DataFrame) -> pd.DataFrame:
    df = coerce_metrics_to_numeric(
        df,
        ["frequency", "monetary_value", "total_volume"]
    )
    
    metrics = ["frequency", "monetary_value", "total_volume"]
    output = []
    
    for metric in metrics:
        overall = (
            df
            .groupby("original_incentive_name")[metric]
            .mean()
            .rename("before")
        )
        
        filtered = (
            df.loc[df["in_correction"] == 1]
            .groupby("original_incentive_name")[metric]
            .mean()
            .rename("after")
        )
        
        combined = (
            pd.concat([overall, filtered], axis=1)
            .assign(metric=metric)
            .reset_index()
        )
        
        output.append(combined)
    
    df_incentive_level = pd.concat(output, ignore_index=True)
    
    # total (not split by incentive)
    total_rows = []
    
    for metric in metrics:
        total_before = df[metric].mean()
        total_after = df.loc[df["in_correction"] == 1, metric].mean()
        
        total_rows.append({
            "original_incentive_name": "TOTAL",
            "before": total_before,
            "after": total_after,
            "metric": metric
        })
    
    df_total = pd.DataFrame(total_rows)
    
    return pd.concat([df_incentive_level, df_total], ignore_index=True)


df_comparison = compute_means_comparison(df)

df_comparison

,original_incentive_name,before,after,metric
0,BNLX_ChurnP_10_test_export.csv,5.564983,5.498936,frequency
1,BNLX_ChurnP_10eu_test_export.csv,4.834343,4.736444,frequency
2,BNLX_ChurnP_250_test_export.csv,4.370034,4.395668,frequency
3,BNLX_ChurnP_25_test_export.csv,3.825589,3.623857,frequency
4,BNLX_ChurnP_500_test_export.csv,4.320875,4.064213,frequency
5,BNLX_ChurnP_5eu_test_export.csv,3.359933,3.299679,frequency
6,BNLX_ChurnP_SKUe_test_export.csv,5.756566,5.613828,frequency
7,BNLX_ChurnP_10_test_export.csv,147.680735,147.349835,monetary_value
8,BNLX_ChurnP_10eu_test_export.csv,114.897205,113.989996,monetary_value
9,BNLX_ChurnP_250_test_export.csv,98.358944,98.038985,monetary_value
